# DESI DR1 browser-representation evidence

This notebook creates the figures and metrics required before building a research poster. It compares observed DESI DR1 LSS rows with the current GPU display policy and two deterministic comparator policies.

**Scientific boundary:** it is a browser-representation validation. It does not estimate clustering, a density field, completeness correction, or physical voids.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repository_url = 'https://github.com/Biswajit1999/NASADIYA-LIGHTCONE.git'
repository_dir = Path('/content/NASADIYA-LIGHTCONE')
if not repository_dir.exists():
    subprocess.run(['git', 'clone', '--depth', '1', repository_url, str(repository_dir)], check=True)
os.chdir(repository_dir)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Repository ready:', Path.cwd())

## Select the local Parquet source

Set `USE_GOOGLE_DRIVE = True` when the 194 MB bundle is stored in Drive. Set it to `False` to upload the file from your computer. Update the Drive path if needed.

In [ ]:
USE_GOOGLE_DRIVE = True
DRIVE_PARQUET_PATH = '/content/drive/MyDrive/desi_dr1_lss_research_bundle.parquet'

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = Path(DRIVE_PARQUET_PATH)
else:
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = Path('/content') / next(iter(uploaded))

MANIFEST_PATH = Path('data/processed/desi-dr1/full-cloud/full-cloud.json')
assert DATA_PATH.exists(), f'Missing Parquet input: {DATA_PATH}'
assert MANIFEST_PATH.exists(), f'Missing full-cloud manifest: {MANIFEST_PATH}'
print('Input:', DATA_PATH)
print('Manifest:', MANIFEST_PATH)

In [ ]:
output_dir = Path('figures/publication_evidence')
command = [
    sys.executable, 'scripts/build_desi_publication_figures.py',
    '--input', str(DATA_PATH),
    '--full-cloud-manifest', str(MANIFEST_PATH),
    '--output-dir', str(output_dir),
    '--budgets', '125000,250000,500000,1000000',
    '--poster-budget', '125000',
    '--dpi', '300',
]
subprocess.run(command, check=True)

In [ ]:
import pandas as pd
from IPython.display import display

metrics = pd.read_csv(output_dir / 'representation_fidelity_metrics.csv')
display(metrics.round(6))
print((output_dir / 'evidence_manifest.json').read_text(encoding='utf-8'))

In [ ]:
from PIL import Image

for image_path in sorted(output_dir.glob('*.png')):
    print(image_path.name)
    display(Image.open(image_path))

In [ ]:
import shutil
archive = shutil.make_archive('/content/publication_evidence', 'zip', output_dir)
from google.colab import files
files.download(archive)